In [1]:
import pandas as pd
import itertools
import random
import numpy as np
from rdkit import Chem
import random
import os
from tqdm import tqdm

In [2]:
def is_valid_smiles(sml):
    try:
        mol = Chem.MolFromSmiles(sml)
        return mol is not None  
    except:
        return False 

In [3]:
#### Step 1: Process the drug information
drug_df = pd.read_csv('raw_data/repo-sample-annotation-20240610.txt', sep='\t')
print(drug_df.columns)
drug_df = drug_df[~drug_df['InChIKey'].astype(str).str.startswith('InChI=')]
drug_df = drug_df[['pubchem_cid', 'InChIKey', 'pert_iname', 'smiles']]
drug_df = drug_df.dropna(subset=['smiles'])
drug_df['smiles'] = drug_df['smiles'].str.split().str[0]
drug_df = drug_df[drug_df['smiles'].apply(is_valid_smiles)].reset_index(drop=True)
drug_df = drug_df.drop_duplicates(subset=['pert_iname']).reset_index(drop=True)

drug_df.columns = ['CID', 'INCHIKEY', 'DRUGNAME', 'SMILES']
drug_df['DRUGID'] = drug_df['INCHIKEY']

# CID 列中非空的值转换为“不带小数点的字符串”，空值保持不变
drug_df['CID'] = drug_df['CID'].apply(lambda x: str(int(x)) if pd.notna(x) else x)

drug_df.to_excel('./process_data/DrugInfo.xlsx', index=False)
print(f"Processed drug_df has {len(drug_df)} rows.")

drug_df

Index(['broad_id', 'pert_iname', 'qc_incompatible', 'purity', 'vendor',
       'catalog_no', 'vendor_name', 'expected_mass', 'smiles', 'InChIKey',
       'pubchem_cid', 'deprecated_broad_id'],
      dtype='object')
Processed drug_df has 7521 rows.


,CID,INCHIKEY,DRUGNAME,SMILES,DRUGID
0,124870683,RWLOGRLTDKDANT-ZPGRZCPFSA-N,ARV-825,Cc1sc-2c(c1C)C(=N[C@@H](CC(=O)Nc1ccc(OCCOCCOCC...,RWLOGRLTDKDANT-ZPGRZCPFSA-N
1,NaN,NaN,3-benzoyl-n-(1-phenylethyl)thiazolidine-4-carb...,C[C@@H](NC(=O)[C@H]1CSCN1C(=O)c1ccccc1)c1ccccc1,NaN
2,135419186,YUFCOOWNNHGGOD-UMMCILCDSA-N,8-bromo-cGMP,Nc1nc(=O)c2nc(Br)n([C@@H]3O[C@@H]4CO[P@](O)(=O...,YUFCOOWNNHGGOD-UMMCILCDSA-N
3,40473171,HNSDLXPSAYFUHK-SQNIBIBYSA-N,docusate,CCCC[C@@H](CC)COC(=O)C[C@@H](C(=O)OC[C@@H](CC)...,HNSDLXPSAYFUHK-SQNIBIBYSA-N
4,NaN,NaN,nifurtimox,CC1CS(=O)(=O)CCN1N=Cc1ccc(o1)[N+]([O-])=O,NaN
...,...,...,...,...,...
7516,NaN,WHBMMWSBFZVSSR-GSVOUGTGSA-N,"d,l-3-hydroxybutyric acid",C[C@@H](O)CC(O)=O,WHBMMWSBFZVSSR-GSVOUGTGSA-N
7517,NaN,DBZQFUNLCALWDY-PNHWDRBUSA-N,3-deazaadenosine (hydrochloride),Nc1nccc2n(cnc12)[C@@H]1O[C@H](CO)[C@@H](O)[C@H]1O,DBZQFUNLCALWDY-PNHWDRBUSA-N
7518,NaN,OCAXFDULERPAJM-UHFFFAOYSA-N,3-cpmt,CN1C2CCC1CC(C2)OC(c1ccccc1)c1ccc(Cl)cc1,OCAXFDULERPAJM-UHFFFAOYSA-N
7519,NaN,MHNSOBBJZCWUGS-UHFFFAOYSA-N,3-alpha-bis(4-fluorophenyl)methoxytropane hydr...,CN1C2CCC1CC(C2)OC(c1ccc(F)cc1)c1ccc(F)cc1,MHNSOBBJZCWUGS-UHFFFAOYSA-N


In [4]:
### Step 2: Process protein information
prot_df = pd.read_excel("../UniprotKB/uniprotkb_2026_04_01.xlsx")

prot_df['Gene Names'] = prot_df['Gene Names'].fillna('').astype(str)

prot_df = (
    prot_df.assign(genename=prot_df['Gene Names'].str.split())  
    .explode('genename')  
    .replace({'genename': {'': None}})  
    .dropna(subset=['genename'])  
    .rename(columns={'genename': 'GeneName'})  
    .reset_index(drop=True)
)

prot_df = prot_df[['Entry', 'GeneName', 'Sequence']]
prot_df.columns = ['UNIPROTID', 'GENENAME', 'SEQUENCE']

prot_df

,UNIPROTID,GENENAME,SEQUENCE
0,A0A087X1C5,CYP2D7,MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...
1,A0A096LP01,SMIM26,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...
2,A0A096LP01,LINC00493,MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...
3,A0A0B4J2F0,PIGBOS1,MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...
4,A0A0C5B5G6,MT-RNR1,MRWQEMGYIFYPRKLR
...,...,...,...
45521,Q9Y4M8,C8orf71,MATFHRAHATSSVKPRARRHQEPNSGDWPGSYRAGTRCSAIGFRLL...
45522,Q9Y6C7,LINC00312,MAHHSLNTFYIWHNNVLHTHLVFFLPHLLNQPFSRGSFLIWLLLCW...
45523,Q9Y6C7,LOH3CR2A,MAHHSLNTFYIWHNNVLHTHLVFFLPHLLNQPFSRGSFLIWLLLCW...
45524,Q9Y6C7,NCRNA00312,MAHHSLNTFYIWHNNVLHTHLVFFLPHLLNQPFSRGSFLIWLLLCW...


In [5]:
### Step 3: Process the information of drug target pairs
pair_df = pd.read_csv('./raw_data/repo-drug-annotation-20200324.txt', sep='\t')
print(pair_df.columns)
pair_df

Index(['pert_iname', 'clinical_phase', 'moa', 'target', 'disease_area',
       'indication'],
      dtype='object')


,pert_iname,clinical_phase,moa,target,disease_area,indication
0,10058-F4,Preclinical,c-Myc inhibitor,NaN,NaN,NaN
1,10-deacetylbaccatin,Preclinical,antitumor agent,NaN,NaN,NaN
2,10-DEBC,Preclinical,AKT inhibitor,PIM1,NaN,NaN
3,10-hydroxycamptothecin,Preclinical,topoisomerase inhibitor,TOP1,NaN,NaN
4,"1,12-Besm",Phase 2,polyamine biosynthesis inhibitor,NaN,NaN,NaN
...,...,...,...,...,...,...
7535,zoxazolamine,Phase 2,myorelaxant,NaN,NaN,NaN
7536,ZSET1446,Preclinical,nicotinic receptor agonist,CACNA1G | CHAT,NaN,NaN
7537,ZSTK-474,Phase 1/Phase 2,PI3K inhibitor,PIK3CA | PIK3CB | PIK3CD | PIK3CG,NaN,NaN
7538,zuclopenthixol,Launched,dopamine receptor antagonist,ADRA1A | ADRA2A | DRD1 | DRD2 | DRD5 | HRH1 | ...,neurology/psychiatry | neurology/psychiatry,bipolar disorder | schizophrenia


In [6]:
expand_pair_df = pair_df.assign(target=pair_df['target'].str.split('|')).explode('target')
expand_pair_df['target'] = expand_pair_df['target'].str.strip()

# 删除 target 为空值（NaN）的行
expand_pair_df = expand_pair_df.dropna(subset=['target']).reset_index(drop=True)
print(f"Processed expand_pair_df has {len(expand_pair_df)} rows.")

expand_pair_df = expand_pair_df[['pert_iname', 'target']]
expand_pair_df.columns = ['DRUGNAME', 'GENENAME']
dti_pos_df = expand_pair_df.copy()

pos_pair_df = pd.merge(dti_pos_df, prot_df, on='GENENAME', how='inner')
pos_pair_df = pd.merge(pos_pair_df, drug_df, on='DRUGNAME', how='inner')

# remove NaN and '' rows
pos_pair_df = pos_pair_df[~((pos_pair_df['DRUGID'].isna()) | (pos_pair_df['DRUGID'] == ''))]
print('Before: ', len(pos_pair_df), pos_pair_df.columns)

# 根据 INCHIKEY 和 UNIPROTID 两列的组合，对 pos_pair_df 去重
pos_pair_df = pos_pair_df.drop_duplicates(subset=['INCHIKEY', 'UNIPROTID']).copy()
print('After: ', len(pos_pair_df))

save_dir = './final_data/'
os.makedirs(save_dir, exist_ok=True)
pos_pair_df.to_excel(save_dir + 'DRH.xlsx', index=False)
# cols = ['DRUGNAME', 'GENENAME', 'UNIPROTID', 'DRUGCID', 'INCHIKEY', 'SMILES', 'SEQUENCE']
# pos_pair_df = pos_pair_df[cols]

pos_pair_df

Processed expand_pair_df has 15964 rows.
Before:  15834 Index(['DRUGNAME', 'GENENAME', 'UNIPROTID', 'SEQUENCE', 'CID', 'INCHIKEY',
       'SMILES', 'DRUGID'],
      dtype='object')
After:  15698


,DRUGNAME,GENENAME,UNIPROTID,SEQUENCE,CID,INCHIKEY,SMILES,DRUGID
0,10-DEBC,PIM1,P11309,MLLSKINSLAHLRAAPCNDLHATKLAPGKEKEPLESQYQVGPLLGS...,10521421,GYBXAGDWMCJZJK-UHFFFAOYSA-N,CCN(CC)CCCCN1c2ccccc2Oc2ccc(Cl)cc12,GYBXAGDWMCJZJK-UHFFFAOYSA-N
1,10-hydroxycamptothecin,TOP1,P11387,MSGDHLHNDSQIEADFRLNDSHKHKDKHKDREHRHKEHKKEKDREK...,97226,HAWSQZCWOQZXHI-FQEVSTJZSA-N,CC[C@@]1(O)C(=O)OCc2c1cc1-c3nc4ccc(O)cc4cc3Cn1...,HAWSQZCWOQZXHI-FQEVSTJZSA-N
2,"1,2,3,4,5,6-hexabromocyclohexane",JAK2,O60674,MGMACLTMTEMEGTSTSSIYQNGDISGNANSMKQIDPVLQVYLYHS...,74603,QFQZKISCBJKVHI-UHFFFAOYSA-N,BrC1C(Br)C(Br)C(Br)C(Br)C1Br,QFQZKISCBJKVHI-UHFFFAOYSA-N
3,12-O-tetradecanoylphorbol-13-acetate,CD4,P01730,MNRGVPFRHLLLVLQLALLPAATQGKKVVLGKKGDTVELTCTASQK...,27924,PHEDXBVPIONUQT-RGYGYFBISA-N,CCCCCCCCCCCCCC(=O)O[C@@H]1[C@@H](C)[C@]2(O)[C@...,PHEDXBVPIONUQT-RGYGYFBISA-N
4,12-O-tetradecanoylphorbol-13-acetate,KCNT2,Q6UVM3,MVDLESEVPPLPPRYRFRDLLLGDQGWQNDDRVQVEFYMNENTFKE...,27924,PHEDXBVPIONUQT-RGYGYFBISA-N,CCCCCCCCCCCCCC(=O)O[C@@H]1[C@@H](C)[C@]2(O)[C@...,PHEDXBVPIONUQT-RGYGYFBISA-N
...,...,...,...,...,...,...,...,...
16426,zuclopenthixol,DRD5,P21918,MLPPGSNGTAYPGQFALYQQLAQGNAVGGSAGAPPLGPSQVVTACL...,5311507,WFPIAZLQTJBIFN-DVZOWYKESA-N,OCCN1CCN(CC\C=C2\c3ccccc3Sc3ccc(Cl)cc23)CC1,WFPIAZLQTJBIFN-DVZOWYKESA-N
16427,zuclopenthixol,HRH1,P35367,MSLPNSSCLLEDKMCEGNKTTMASPQLMPLVVVLSTICLVTVGLNL...,5311507,WFPIAZLQTJBIFN-DVZOWYKESA-N,OCCN1CCN(CC\C=C2\c3ccccc3Sc3ccc(Cl)cc23)CC1,WFPIAZLQTJBIFN-DVZOWYKESA-N
16428,zuclopenthixol,HTR2A,P28223,MDILCEENTSLSSTTNSLMQLNDDTRLYSNDFNSGEANTSDAFNWT...,5311507,WFPIAZLQTJBIFN-DVZOWYKESA-N,OCCN1CCN(CC\C=C2\c3ccccc3Sc3ccc(Cl)cc23)CC1,WFPIAZLQTJBIFN-DVZOWYKESA-N
16429,zuranolone,GABRA1,P14867,MRKSPGLSDCLWAWILLLSTLTGRSYGQPSLQDELKDNTTVFTRIL...,86294073,HARRKNSQXBRBGZ-GVKWWOCJSA-N,[H][C@@]12CC[C@H](C(=O)Cn3cc(cn3)C#N)[C@@]1(C)...,HARRKNSQXBRBGZ-GVKWWOCJSA-N


In [7]:
pos_drugs = list(pos_pair_df["DRUGID"].unique().tolist())
pos_genes = list(pos_pair_df["UNIPROTID"].unique().tolist())

print(f"Number of positive drugs: {len(pos_drugs)}")
print(f"Number of positive prots: {len(pos_genes)}")

Number of positive drugs: 5238
Number of positive prots: 2444


In [8]:
import pandas as pd
import random
from itertools import product

# 读取数据
pair_df = pd.read_excel(r'./final_data/DRH.xlsx')
print('Before filtering, number of pairs:', len(pair_df))

# ['DRUGNAME', 'GENENAME', 'UNIPROTID', 'SEQUENCE', 'DRUGCID', 'INCHIKEY', 'SMILES']

# ===== 1. 构建字典 =====
drug_df = pair_df[['DRUGID', 'INCHIKEY', 'CID', 'DRUGNAME', 'SMILES']].drop_duplicates()

protein_df = pair_df[['UNIPROTID', 'GENENAME', 'SEQUENCE']].drop_duplicates()

drug_dict = dict(zip(drug_df['DRUGID'], drug_df['SMILES']))
protein_dict = dict(zip(protein_df['UNIPROTID'], protein_df['SEQUENCE']))

print('Number of drugs:', len(drug_dict))
print('Number of proteins:', len(protein_dict))

# 保存字典
drug_df.to_excel(r'./final_data/DrugInfo.xlsx', index=False)
protein_df.to_excel(r'./final_data/ProtInfo.xlsx', index=False)

# ===== 2. 正样本 =====
pos_pairs = set(zip(pair_df['DRUGID'], pair_df['UNIPROTID']))
num_pos = len(pos_pairs)
print('Number of positive pairs:', num_pos)

# ===== 3. 生成全部负样本 =====
all_drugs = pair_df['DRUGID'].dropna().unique()
all_proteins = pair_df['UNIPROTID'].dropna().unique()

print('Generating all candidate negative pairs...')
all_pairs = set(product(all_drugs, all_proteins))
all_neg_pairs = list(all_pairs - pos_pairs)

print('Total candidate negative pairs:', len(all_neg_pairs))

ratios = [1, 3, 5, 7, 10]
seeds = [1, 11, 111, 1111, 11111]

for ratio in ratios:
    print(f'Generating dataset with ratio 1:{ratio}...')
    save_dir = f'./final_data/1_{ratio}'
    os.makedirs(save_dir, exist_ok=True)

    num_neg = num_pos * ratio

    for i, seed in enumerate(seeds):
        print(f'Generating dataset {i+1} with seed {seed}...')
        random.seed(seed)
        sampled_neg = random.sample(all_neg_pairs, num_neg)

        neg_df = pd.DataFrame(sampled_neg, columns=['DRUGID', 'UNIPROTID'])
        neg_df['Label'] = 0

        pos_df = pd.DataFrame(list(pos_pairs), columns=['DRUGID', 'UNIPROTID'])
        pos_df['Label'] = 1

        dataset = pd.concat([pos_df, neg_df], ignore_index=True)
        dataset = dataset.sample(frac=1, random_state=seed).reset_index(drop=True)
        dataset.to_excel(os.path.join(save_dir, f'Set{i+1}.xlsx'), index=False)

print('All datasets generated!')

Before filtering, number of pairs: 15698
Number of drugs: 5238
Number of proteins: 2444
Number of positive pairs: 15698
Generating all candidate negative pairs...
Total candidate negative pairs: 12785974
Generating dataset with ratio 1:1...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:3...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:5...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Generating dataset 3 with seed 111...
Generating dataset 4 with seed 1111...
Generating dataset 5 with seed 11111...
Generating dataset with ratio 1:7...
Generating dataset 1 with seed 1...
Generating dataset 2 with seed 11...
Gener